In [0]:
storage_account_name = "azureetlstorage"

storage_account_key = dbutils.secrets.get(
    scope = "etl-scope",
    key   = "adls-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

print("connected!")

In [0]:
# Import audit logger
import sys
sys.path.insert(0, "/Workspace/Users/raojyothirmayee21@gmail.com/utils")
from audit_logger import AuditLogger

# Initialize logger
logger = AuditLogger(
    spark=spark,
    storage_account=storage_account_name,
    pipeline_name="PL_ETL_Pipeline",
    notebook_name="bronze_orders_cust_prod"
)

print("Audit logger initialized!")

In [0]:
# Define the path to your orders CSV in ADLS
orders_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/landing/orders.csv"

# Read CSV into a Spark DataFrame
orders_df = spark.read.format("csv").option("header", "true").load(orders_path)

# Show first 5 rows
orders_df.show(5)

In [0]:
# How many rows?
print(f"Total rows: {orders_df.count()}")

# What columns do we have?
orders_df.printSchema()

In [0]:
# Define Bronze path
bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/orders"

# Write as Delta table
orders_df.write.format("delta").mode("overwrite").save(bronze_path)

print("Bronze orders table written successfully!")

In [0]:
from pyspark.sql import functions as F

# Add metadata columns
bronze_df = orders_df \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_batch_id", F.lit("batch_001"))

# Overwrite with metadata included
bronze_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(bronze_path)

print(f"Bronze orders with metadata written!")
bronze_df.show(3)

In [0]:
# Read back the Bronze table and show
verify_df = spark.read.format("delta").load(bronze_path)
verify_df.show(3)

Product Table Ingestion in Bronze

In [0]:
products_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/landing/products.json"

products_df = spark.read.format("json") \
    .option("multiLine", "true") \
    .load(products_path)

print(f"Total products: {products_df.count()}")
products_df.printSchema()

In [0]:
# Add metadata
products_bronze_df = products_df \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_batch_id", F.lit("batch_001"))

# Write to Bronze
products_bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/products"

products_bronze_df.write.format("delta") \
    .mode("overwrite") \
    .save(products_bronze_path)

print("Bronze products table written!")

Customers Table Ingestion to Bronze

In [0]:
# Read customers CSV from ADLS
customers_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/landing/customers.csv"
customers_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(customers_path)

print(f"Total customers: {customers_df.count()}")
customers_df.printSchema()

In [0]:
from pyspark.sql import functions as F
# Add metadata
customers_bronze_df = customers_df \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name()) \
    .withColumn("_batch_id", F.lit("batch_001"))

# Write to Bronze
customers_bronze_path = f"abfss://datalake@{storage_account_name}.dfs.core.windows.net/bronze/customers"

customers_bronze_df.write.format("delta") \
    .mode("overwrite") \
    .save(customers_bronze_path)

print("Bronze customers table written!")
customers_bronze_df.show(3)